# Access HuggingFace repo, pipe that boy through RAG (Phase 2)

In [ ]:
!pip install -U python-dotenv huggingface_hub datasets

In [2]:
import os
from huggingface_hub import login, whoami
from pathlib import Path

In [ ]:
SECRET_NAME = "HF_TOKEN"        # key name in Colab Secrets and in the env file
ENV_FILE = "public.env"         # create a public.env with key "HF_TOKEN" in the same folder

def token_from_colab(name: str) -> str:
    from google.colab import userdata          # raises ImportError outside Colab
    return userdata.get(name)                  # raises if secret is missing or access is denied


def token_from_envfile(name: str, path: str) -> str:
    from dotenv import dotenv_values

    env_path = Path.cwd() / path
    if not env_path.is_file():
        raise FileNotFoundError(f"{env_path} does not exist. Did you forget to create the file?")

    value = dotenv_values(env_path).get(name)
    if not value or not value.strip():
        raise KeyError(f"'{name}' is missing or blank in {env_path}. Did you forget to add the variables?")
    return value.strip()


HF_TOKEN = None

# 1) Colab Secret
try:
    HF_TOKEN = token_from_colab(SECRET_NAME)
except Exception as e:
    print(f"Colab Secret not available ({type(e).__name__}), trying {ENV_FILE}...")

# 2) Fall back to the .env file (raises its own error if this fails too)
if not HF_TOKEN:
    HF_TOKEN = token_from_envfile(SECRET_NAME, ENV_FILE)

login(token=HF_TOKEN)
print(whoami())
os.environ["HF_TOKEN"] = HF_TOKEN

In [4]:
from datasets import load_dataset

In [ ]:
repo_id = "<REPO_ID>"

dataset_dict = load_dataset(repo_id, token=HF_TOKEN)
train_df = dataset_dict["train"].to_pandas()
test_df = dataset_dict["test"].to_pandas()

DISEASE_LABELS = sorted(train_df["disease"].unique())

print(f"train cases: {len(train_df)} | test cases: {len(test_df)} | diseases: {len(DISEASE_LABELS)}")
train_df.head()